# 01 · Pandas 101 : les bases

La question du jour : **combien de courses de taxi par *borough* (arrondissement de New York) en mars 2026, et combien coûtent-elles en moyenne ?**
On y répond pas à pas avec **pandas**, la bibliothèque Python qui manipule des tableaux de données.

## 1. Lire un fichier CSV

On importe pandas (`pd`) et on note les dossiers `DATA` (données) et `OUTPUT` (résultats). `pd.read_csv` lit un fichier CSV et renvoie un **DataFrame** : un tableau avec des lignes et des colonnes.

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("../data")
OUTPUT = Path("../output")
OUTPUT.mkdir(exist_ok=True)

zones = pd.read_csv(DATA / "taxi_zone_lookup.csv")
zones.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


`head()` affiche les 5 premières lignes. Chaque ligne est une zone de taxi : son numéro (`LocationID`), son *borough* et son nom.

## 2. Taille, colonnes et types

`shape` donne (nombre de lignes, nombre de colonnes), `columns` les noms des colonnes et `dtypes` leur type.

In [2]:
print(zones.shape)
print(list(zones.columns))
zones.dtypes

(265, 4)
['LocationID', 'Borough', 'Zone', 'service_zone']


LocationID      int64
Borough           str
Zone              str
service_zone      str
dtype: object

La table compte 265 zones et 4 colonnes. `int64` signifie nombre entier, `str` signifie texte.

## 3. Choisir des colonnes

Avec un nom de colonne entre crochets, on obtient une **Series** : une seule colonne.

In [3]:
zones["Borough"].head()

0              EWR
1           Queens
2            Bronx
3        Manhattan
4    Staten Island
Name: Borough, dtype: str

Avec une **liste** de noms (doubles crochets), on obtient un DataFrame avec seulement ces colonnes.

In [4]:
zones[["Zone", "Borough"]].head()

,Zone,Borough
0,Newark Airport,EWR
1,Jamaica Bay,Queens
2,Allerton/Pelham Gardens,Bronx
3,Alphabet City,Manhattan
4,Arden Heights,Staten Island


## 4. Filtrer et compter

`zones["Borough"] == "Manhattan"` vaut `True` ou `False` pour chaque ligne ; placé entre crochets, il ne garde que les lignes à `True`.

In [5]:
manhattan = zones[zones["Borough"] == "Manhattan"]
print(len(manhattan))
manhattan.head()

69


,LocationID,Borough,Zone,service_zone
3,4,Manhattan,Alphabet City,Yellow Zone
11,12,Manhattan,Battery Park,Yellow Zone
12,13,Manhattan,Battery Park City,Yellow Zone
23,24,Manhattan,Bloomingdale,Yellow Zone
40,41,Manhattan,Central Harlem,Boro Zone


Manhattan compte 69 zones. Pour compter les lignes de **chaque** valeur d'une colonne, il y a `value_counts()`.

In [6]:
zones["Borough"].value_counts()

Borough
Queens           69
Manhattan        69
Brooklyn         61
Bronx            43
Staten Island    20
EWR               1
Unknown           1
Name: count, dtype: int64

Le Queens et Manhattan ont 69 zones chacun ; `EWR` est l'aéroport de Newark. La 265e zone (« Outside of NYC ») n'a pas de *borough* : `value_counts()` ignore les cases vides.

## 5. Lire les courses (Parquet)

Les courses sont dans un fichier **Parquet** : on le lit avec `pd.read_parquet`. `%%time`, placé en première ligne, mesure la durée de la cellule (*Wall time* = durée réelle écoulée).

In [7]:
%%time
courses = pd.read_parquet(DATA / "yellow_tripdata_2026-03.parquet")
courses.shape

CPU times: user 1.19 s, sys: 256 ms, total: 1.44 s
Wall time: 250 ms


(3952451, 20)

Près de 4 millions de courses (3 952 451) et 20 colonnes : la lecture ne prend qu'un instant (voir *Wall time*).

In [8]:
courses.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-03-01 00:02:26,2026-03-01 00:13:45,1.0,2.58,1.0,N,48,151,1,14.2,1.0,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75
1,2,2026-03-01 00:19:33,2026-03-01 00:28:21,1.0,1.50,1.0,N,238,166,1,10.0,1.0,0.5,0.00,0.0,1.0,15.00,2.5,0.0,0.00
2,2,2026-03-01 00:07:20,2026-03-01 00:15:12,2.0,0.88,1.0,N,90,249,1,8.6,1.0,0.5,2.87,0.0,1.0,17.22,2.5,0.0,0.75
3,2,2026-03-01 00:16:11,2026-03-01 00:28:20,1.0,1.76,1.0,N,249,137,1,12.1,1.0,0.5,3.57,0.0,1.0,21.42,2.5,0.0,0.75
4,2,2026-03-01 00:20:47,2026-03-01 00:30:44,2.0,1.57,1.0,N,100,142,1,11.4,1.0,0.5,3.43,0.0,1.0,20.58,2.5,0.0,0.75


Une ligne = une course : heures de prise en charge et de dépose, distance en miles, zones de départ (`PULocationID`) et d'arrivée (`DOLocationID`), montants en dollars.

In [9]:
courses.dtypes

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object

Les dates sont déjà de vraies dates (`datetime64`) : un fichier Parquet enregistre le type de chaque colonne. Le CSV, lui, ne le fait pas (notebook 02).

## 6. Repérer les valeurs étranges

`describe()` résume des colonnes de nombres : moyenne, minimum, quartiles (25 %, 50 %, 75 %) et maximum.

In [10]:
courses[["trip_distance", "total_amount", "tip_amount"]].describe().round(2)

,trip_distance,total_amount,tip_amount
count,3952451.00,3952451.00,3952451.00
mean,5.98,30.10,2.86
std,581.05,22.41,3.91
min,0.00,-1022.46,-75.75
25%,1.01,17.22,0.00
50%,1.82,23.40,2.11
75%,3.71,34.90,3.95
max,288381.68,1850.55,397.00


La distance médiane (ligne 50 %) est de 1,8 mile et le montant médian de 23,40 $. Mais on voit des montants négatifs (−1 022 $) et une distance de 288 382 miles : il y a des erreurs dans les données.

## 7. Créer une colonne

Soustraire deux dates donne une durée ; `.dt.total_seconds()` la convertit en secondes. Pour créer une colonne, il suffit d'écrire `courses["nouveau_nom"] = ...`.

In [11]:
duree = courses["tpep_dropoff_datetime"] - courses["tpep_pickup_datetime"]
courses["duree_min"] = duree.dt.total_seconds() / 60
courses[["tpep_pickup_datetime", "tpep_dropoff_datetime", "duree_min"]].head()

,tpep_pickup_datetime,tpep_dropoff_datetime,duree_min
0,2026-03-01 00:02:26,2026-03-01 00:13:45,11.316667
1,2026-03-01 00:19:33,2026-03-01 00:28:21,8.800000
2,2026-03-01 00:07:20,2026-03-01 00:15:12,7.866667
3,2026-03-01 00:16:11,2026-03-01 00:28:20,12.150000
4,2026-03-01 00:20:47,2026-03-01 00:30:44,9.950000


La première course a duré un peu plus de 11 minutes. Le calcul est fait d'un coup sur les 4 millions de lignes, sans boucle.

## 8. Trier

`sort_values` trie le tableau selon une colonne ; `ascending=False` met les plus grandes valeurs en premier.

In [12]:
colonnes = ["tpep_pickup_datetime", "trip_distance", "total_amount", "payment_type"]
courses[colonnes].sort_values("total_amount", ascending=False).head()

,tpep_pickup_datetime,trip_distance,total_amount,payment_type
434248,2026-03-05 17:15:25,205.86,1850.55,2
2591980,2026-03-27 15:42:02,5.61,1441.35,3
2110598,2026-03-22 11:33:51,167.82,1157.41,2
1091405,2026-03-12 12:11:29,238.35,1022.46,4
2760453,2026-03-29 04:17:38,0.40,951.00,2


La course la plus chère coûte 1 850 $ pour 206 miles. La cinquième coûte 951 $ pour 0,4 mile : sans doute une erreur de saisie.

## 9. Grouper

`groupby` rassemble les lignes qui ont la même valeur, puis calcule un résultat par groupe. Codes de `payment_type` : 1 = carte, 2 = espèces, 3 = gratuit, 4 = litige, 0 = non renseigné (courses commandées par application).

In [13]:
courses.groupby("payment_type")["total_amount"].mean().round(2)

payment_type
0    34.08
1    29.87
2    24.18
3    12.72
4     2.99
Name: total_amount, dtype: float64

Avec `agg`, on calcule plusieurs résultats d'un coup ; le nom donné devient le nom de la colonne.

In [14]:
par_paiement = courses.groupby("payment_type")["total_amount"].agg(nb_courses="count", montant_moyen="mean")
par_paiement.round(2)

,nb_courses,montant_moyen
payment_type,,
0,945748,34.08
1,2608830,29.87
2,353541,24.18
3,12842,12.72
4,31490,2.99


La carte domine : 2,6 millions de courses, à 29,87 $ en moyenne. Les courses payées en espèces sont moins nombreuses (354 000) et moins chères (24,18 $).

## 10. Joindre deux tables

Les courses ne contiennent que le **numéro** de la zone de départ (`PULocationID`). `merge` ajoute à chaque course la ligne de `zones` qui a le même numéro (`LocationID`), comme une jointure SQL.

In [15]:
avec_zones = courses.merge(zones, left_on="PULocationID", right_on="LocationID")
avec_zones[["PULocationID", "LocationID", "Borough", "Zone"]].head()

,PULocationID,LocationID,Borough,Zone
0,48,48,Manhattan,Clinton East
1,238,238,Manhattan,Upper West Side North
2,90,90,Manhattan,Flatiron
3,249,249,Manhattan,West Village
4,100,100,Manhattan,Garment District


Chaque course a maintenant le *borough* et le nom de sa zone de départ. Il reste à grouper par *borough*, comme le « panier moyen par ville » du cours (slide 15) ; `reset_index()` remet `Borough` en colonne normale.

In [16]:
par_borough = avec_zones.groupby("Borough")["total_amount"].agg(nb_courses="count", montant_moyen="mean")
par_borough = par_borough.sort_values("nb_courses", ascending=False).reset_index().round(2)
par_borough

,Borough,nb_courses,montant_moyen
0,Manhattan,3391948,26.01
1,Queens,353174,65.01
2,Brooklyn,160128,36.66
3,Bronx,39925,37.88
4,Unknown,4624,31.50
5,EWR,527,102.22
6,Staten Island,500,43.34


Voici la réponse : Manhattan concentre 86 % des courses (3,4 millions), à 26 $ en moyenne. Les départs du Queens coûtent bien plus cher (65 $) : c'est là que se trouvent les aéroports JFK et LaGuardia.

## 11. Enregistrer le résultat

`to_csv` écrit le tableau dans un fichier ; `index=False` évite d'écrire la colonne des numéros de ligne. On relit le fichier pour vérifier.

In [17]:
par_borough.to_csv(OUTPUT / "courses_par_borough.csv", index=False)
pd.read_csv(OUTPUT / "courses_par_borough.csv")

,Borough,nb_courses,montant_moyen
0,Manhattan,3391948,26.01
1,Queens,353174,65.01
2,Brooklyn,160128,36.66
3,Bronx,39925,37.88
4,Unknown,4624,31.50
5,EWR,527,102.22
6,Staten Island,500,43.34


On retrouve le même tableau : près de 4 millions de courses résumées en 7 lignes.

## Exercices

### ✍️ Exercice 1 — Les longues courses

Combien de courses font plus de 20 miles, et quel est leur montant moyen ?

In [18]:
# ✅ Solution
longues = courses[courses["trip_distance"] > 20]
print("Nombre de courses :", len(longues))
print("Montant moyen :", round(longues["total_amount"].mean(), 2))

Nombre de courses : 33428
Montant moyen : 109.19


Environ 33 000 courses font plus de 20 miles ; elles coûtent 109 $ en moyenne, soit plus de 3 fois la moyenne générale (30 $).

### ✍️ Exercice 2 — Les zones de départ les plus fréquentes

Affichez les 5 zones de départ (nom de la zone) qui ont le plus de courses. Indice : utilisez `avec_zones`.

In [19]:
# ✅ Solution
avec_zones["Zone"].value_counts().head()

Zone
Upper East Side South    174403
Midtown Center           165350
Upper East Side North    157296
JFK Airport              153791
Midtown East             121946
Name: count, dtype: int64

L'Upper East Side et Midtown (Manhattan) dominent ; l'aéroport JFK arrive en 4e position.

### ✍️ Exercice 3 (facultatif) — Le pourboire selon le jour

Calculez le pourboire moyen (`tip_amount`) par jour de la semaine. Indice : `.dt.dayofweek` donne le jour (0 = lundi, 6 = dimanche).

In [20]:
# ✅ Solution
courses["jour"] = courses["tpep_pickup_datetime"].dt.dayofweek
courses.groupby("jour")["tip_amount"].mean().round(2)

jour
0    3.01
1    2.97
2    3.04
3    2.91
4    2.80
5    2.52
6    2.75
Name: tip_amount, dtype: float64

Le pourboire moyen est le plus élevé en semaine (environ 3 $ du lundi au mercredi) et le plus bas le samedi (2,52 $).

## À retenir

- `pd.read_csv` et `pd.read_parquet` renvoient un **DataFrame** (un tableau) ; une colonne seule est une **Series**.
- `head()`, `shape`, `dtypes` et `describe()` : le premier coup d'œil sur des données.
- `df[condition]` filtre, `sort_values` trie, `value_counts` compte.
- `groupby(...)[...].agg(...)` résume par groupe ; `merge` joint deux tables.
- `to_csv(..., index=False)` enregistre le résultat.